# FHIR Patient Silver Lakeflow Transformation

## Purpose

I use this notebook as the transformation source for the FHIR Silver Lakeflow
pipeline.

I keep the Patient Bronze-to-Silver transformation logic that I previously
developed and tested, while Lakeflow now manages the Silver output and applies
the approved Patient quality contract during refresh.

### Source

`health_insurance.bronze.fhir_patient_raw`

### Pipeline target

`health_insurance.silver.fhir_patient`

### Production changes

In this pipeline version:

- I keep the null-safe FHIR parsing and conformance logic I already validated.
- I use an explicit minimal FHIR Patient schema instead of inferring the schema
  at runtime.
- I retain FHIR `meta.versionId` and `meta.lastUpdated` as technical metadata.
- I keep the latest available version of each Patient resource.
- I load WARN, DROP, and FAIL rules dynamically from Unity Catalog.
- I let Lakeflow manage Silver persistence and quality metrics.

The FHIR Bronze table is cumulative and incrementally populated by Auto Loader.
I therefore use a Lakeflow materialized view over the current Bronze state so I
can resolve the latest Patient version deterministically.


In [0]:
# importing the Lakeflow API, Spark functions, schemas, and window support.

from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

CATALOG = "health_insurance"

SOURCE_TABLE = f"{CATALOG}.bronze.fhir_patient_raw"
QUALITY_RULES_TABLE = f"{CATALOG}.governance.quality_rules"


In [0]:
# loading the active Patient quality contract from Unity Catalog.

def get_quality_rules(dataset, severity):
    rows = (
        spark.read
        .table(QUALITY_RULES_TABLE)
        .filter(
            (F.col("dataset") == dataset)
            & (F.col("severity") == severity)
            & F.col("is_active")
        )
        .select(
            "rule_name",
            "constraint"
        )
        .collect()
    )

    return {
        row["rule_name"]: row["constraint"]
        for row in rows
    }


PATIENT_WARN_RULES = get_quality_rules("patient", "WARN")
PATIENT_DROP_RULES = get_quality_rules("patient", "DROP")
PATIENT_FAIL_RULES = get_quality_rules("patient", "FAIL")


## Explicit FHIR Patient schema

During development I inferred the combined Patient schema from Bronze.

For the pipeline version, I define only the FHIR structures that my Silver
transformation actually consumes. This makes the production transformation
more deterministic while still allowing optional FHIR fields to remain NULL.

I also include the FHIR resource metadata required for latest-version handling.


In [0]:
#  defining the minimal FHIR Patient schema required by this transformation.

coding_schema = T.StructType([
    T.StructField("system", T.StringType(), True),
    T.StructField("code", T.StringType(), True),
    T.StructField("display", T.StringType(), True)
])

identifier_type_schema = T.StructType([
    T.StructField(
        "coding",
        T.ArrayType(coding_schema),
        True
    )
])

identifier_schema = T.StructType([
    T.StructField("value", T.StringType(), True),
    T.StructField("type", identifier_type_schema, True)
])

name_schema = T.StructType([
    T.StructField("use", T.StringType(), True),
    T.StructField(
        "given",
        T.ArrayType(T.StringType()),
        True
    ),
    T.StructField("family", T.StringType(), True)
])

address_schema = T.StructType([
    T.StructField(
        "line",
        T.ArrayType(T.StringType()),
        True
    ),
    T.StructField("city", T.StringType(), True),
    T.StructField("state", T.StringType(), True),
    T.StructField("postalCode", T.StringType(), True),
    T.StructField("country", T.StringType(), True)
])

telecom_schema = T.StructType([
    T.StructField("system", T.StringType(), True),
    T.StructField("value", T.StringType(), True)
])

meta_schema = T.StructType([
    T.StructField("versionId", T.StringType(), True),
    T.StructField("lastUpdated", T.StringType(), True)
])

patient_schema = T.StructType([
    T.StructField("id", T.StringType(), True),
    T.StructField("gender", T.StringType(), True),
    T.StructField("birthDate", T.StringType(), True),
    T.StructField(
        "identifier",
        T.ArrayType(identifier_schema),
        True
    ),
    T.StructField(
        "name",
        T.ArrayType(name_schema),
        True
    ),
    T.StructField(
        "address",
        T.ArrayType(address_schema),
        True
    ),
    T.StructField(
        "telecom",
        T.ArrayType(telecom_schema),
        True
    ),
    T.StructField("meta", meta_schema, True)
])


## Reusable Patient transformation

I keep the transformation itself separate from the Lakeflow dataset definition.

The function performs the same Patient parsing, safe array access, identifier
selection, demographic standardization, and age derivation I validated in the
development notebook. I additionally retain FHIR version metadata and resolve
the latest resource version before publishing Silver.


In [0]:
# applying the validated FHIR Patient Bronze-to-Silver transformation.

def transform_patient(patient_bronze_df):

    patient_parsed_df = (
        patient_bronze_df
        .withColumn(
            "_record_hash",
            F.sha2(F.col("raw_json"), 256)
        )
        .withColumn(
            "patient",
            F.from_json(
                F.col("raw_json"),
                patient_schema
            )
        )
    )

    patient_core_df = (
        patient_parsed_df
        .select(
            F.col("patient.id").alias("patient_id"),
            F.col("patient.gender").alias("gender"),
            F.to_date(
                F.col("patient.birthDate")
            ).alias("birth_date"),

            F.col("patient.name").alias("name"),
            F.col("patient.address").alias("address"),
            F.col("patient.telecom").alias("telecom"),
            F.col("patient.identifier").alias("identifier"),

            F.col("patient.meta.versionId").alias(
                "fhir_version_id"
            ),
            F.to_timestamp(
                F.col("patient.meta.lastUpdated")
            ).alias("fhir_last_updated"),

            "_record_hash",
            "_ingested_at",
            "_source_system",
            "_resource_type"
        )
    )

    patient_name_df = (
        patient_core_df

        .withColumn(
            "official_names",
            F.expr("""
                filter(
                    name,
                    x -> x.use = 'official'
                )
            """)
        )

        .withColumn(
            "official_name",
            F.expr("get(official_names, 0)")
        )

        .withColumn(
            "given_name",
            F.expr("get(official_name.given, 0)")
        )

        .withColumn(
            "family_name",
            F.col("official_name.family")
        )
    )

    patient_address_df = (
        patient_name_df

        .withColumn(
            "primary_address",
            F.expr("get(address, 0)")
        )

        .withColumn(
            "address_line",
            F.expr("get(primary_address.line, 0)")
        )

        .withColumn(
            "city",
            F.col("primary_address.city")
        )

        .withColumn(
            "state",
            F.col("primary_address.state")
        )

        .withColumn(
            "postal_code",
            F.col("primary_address.postalCode")
        )

        .withColumn(
            "country",
            F.col("primary_address.country")
        )
    )

    patient_contact_df = (
        patient_address_df

        .withColumn(
            "phone_contacts",
            F.expr("""
                filter(
                    telecom,
                    x -> x.system = 'phone'
                )
            """)
        )

        .withColumn(
            "phone",
            F.expr("get(phone_contacts.value, 0)")
        )
    )

    patient_identifier_df = (
        patient_contact_df

        .withColumn(
            "medical_record_identifiers",
            F.expr("""
                filter(
                    identifier,
                    x -> exists(
                        x.type.coding,
                        c -> c.code = 'MR'
                    )
                )
            """)
        )

        .withColumn(
            "medical_record_number",
            F.expr(
                "get(medical_record_identifiers.value, 0)"
            )
        )
    )

    patient_enriched_df = (
        patient_identifier_df

        .withColumn(
            "age",
            F.floor(
                F.months_between(
                    F.current_date(),
                    F.col("birth_date")
                ) / 12
            ).cast("int")
        )

        .withColumn(
            "age_group",
            F.when(
                F.col("age").isNull(),
                "UNKNOWN"
            )
            .when(
                F.col("age") < 18,
                "UNDER_18"
            )
            .when(
                F.col("age") < 35,
                "18_34"
            )
            .when(
                F.col("age") < 50,
                "35_49"
            )
            .when(
                F.col("age") < 65,
                "50_64"
            )
            .otherwise("65_PLUS")
        )
    )

    patient_standardized_df = (
        patient_enriched_df

        .withColumn(
            "gender",
            F.upper(F.trim("gender"))
        )

        .withColumn(
            "city",
            F.initcap(F.trim("city"))
        )

        .withColumn(
            "state",
            F.upper(F.trim("state"))
        )

        .withColumn(
            "country",
            F.upper(F.trim("country"))
        )
    )

    patient_conformed_df = (
        patient_standardized_df

        .select(
            "patient_id",
            "medical_record_number",

            "given_name",
            "family_name",

            "gender",
            "birth_date",
            "age",
            "age_group",

            "phone",

            "address_line",
            "city",
            "state",
            "postal_code",
            "country",

            "fhir_version_id",
            "fhir_last_updated",

            "_record_hash",
            "_source_system",
            "_resource_type",
            "_ingested_at"
        )

        .withColumn(
            "_silver_transformed_at",
            F.current_timestamp()
        )
    )

    patient_versioned_df = (
        patient_conformed_df
        .withColumn(
            "_dedup_key",
            F.coalesce(
                F.col("patient_id"),
                F.col("_record_hash")
            )
        )
    )

    latest_patient_window = (
        Window
        .partitionBy("_dedup_key")
        .orderBy(
            F.col("fhir_last_updated").desc_nulls_last(),
            F.col("_ingested_at").desc_nulls_last(),
            F.col("fhir_version_id").desc_nulls_last()
        )
    )

    patient_latest_df = (
        patient_versioned_df

        .withColumn(
            "_version_rank",
            F.row_number().over(latest_patient_window)
        )

        .filter(
            F.col("_version_rank") == 1
        )

        .drop(
            "_version_rank",
            "_dedup_key",
            "_record_hash"
        )
    )

    return patient_latest_df


## Lakeflow-managed Patient Silver dataset

I attach the centrally governed Patient expectations directly to the Silver
materialized view.

- WARN rules preserve the record and expose quality metrics.
- DROP rules remove unusable Patient records from validated Silver.
- FAIL rules can stop the Patient flow if critical rules are activated later.

The materialized view reads the cumulative Bronze state, resolves the latest
FHIR Patient version, and publishes one current Silver record per Patient.


In [0]:
# I am defining the pipeline-managed FHIR Patient Silver materialized view.

@dp.materialized_view(
    name="fhir_patient",
    comment="Validated and latest-version FHIR Patient records."
)
@dp.expect_all(PATIENT_WARN_RULES)
@dp.expect_all_or_drop(PATIENT_DROP_RULES)
@dp.expect_all_or_fail(PATIENT_FAIL_RULES)
def fhir_patient():

    patient_bronze_df = spark.read.table(
        SOURCE_TABLE
    )

    return transform_patient(
        patient_bronze_df
    )


## Pipeline result

This notebook no longer writes `health_insurance.silver.fhir_patient`
manually.

When I add it as a source to the FHIR Silver Lakeflow pipeline:

1. Lakeflow reads the cumulative FHIR Patient Bronze table.
2. I parse the Patient JSON with an explicit production schema.
3. I apply the previously validated Patient transformation logic.
4. I retain FHIR version metadata and select the latest resource version.
5. Lakeflow evaluates the active governed Patient quality rules.
6. Lakeflow manages the Silver materialized view and expectation metrics.

The development-only schema inference, `display()`, profiling counts, direct
Delta overwrite, and post-write verification cells are no longer part of the
pipeline execution path.
